# N-CMAPSS Baseline — PRAMANA RUL Benchmark

**Why this notebook exists** (from BE-2-START-HERE.md, Task 1):
> This is the highest-value 90 minutes of your week and it is task one for a reason.
> It is **the only thing that proves our ML is not memorising our own simulator.**
> A DRDO panel will ask *"your data is synthetic, why should we believe any of it?"*
> and the answer ends with *"and here it is running unmodified on NASA's flight-condition data, with the score."*

**Dataset:** N-CMAPSS (Chao et al., NASA Prognostics Data Repository) — turbofan run-to-failure under real flight conditions.

Download: https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/
File: `N-CMAPSS_DS01-005.h5` (or any DS0x file)

**Metrics reported (field standard, NOT just RMSE):**
1. RMSE
2. NASA asymmetric score: `Σ exp(-d/13) - 1` if d < 0 (early), `Σ exp(d/10) - 1` if d ≥ 0 (late)
3. Prognostic horizon
4. α–λ accuracy
5. False alarm rate proxy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path

# Point this at the downloaded N-CMAPSS HDF5 file
# Extracted from the NASA PCoE archive (see README / metrics.md §1).
# 2.7 GB, gitignored — re-extract rather than commit it.
NCMAPSS_PATH = Path(__file__).parent / 'data/data_set/N-CMAPSS_DS01-005.h5' \
    if '__file__' in dir() else Path('data/data_set/N-CMAPSS_DS01-005.h5')
DEVICE = 'cpu'
WINDOW = 32
print('Device:', DEVICE)

## 1. Load N-CMAPSS

In [ ]:
def _names(f, key):
    """Decode the *_var name arrays (stored as bytes) into str column names."""
    return [n.decode() if isinstance(n, bytes) else str(n)
            for n in np.array(f[key]).ravel()]


def load_ncmapss(path: Path, split: str = "dev", max_units: int = 6, stride: int = 10):
    """
    Load N-CMAPSS in its ACTUAL on-disk layout, without exhausting memory.

    The real file (Chao, Kulkarni & Goebel 2021) does NOT contain 'train'/'test'
    groups of named columns. It stores flat parallel arrays:

        W_dev / W_test      operating conditions      (n, 4)
        X_s_dev / X_s_test  MEASURED sensors          (n, 14)
        X_v_dev / X_v_test  virtual sensors           (n, 14)
        T_dev  / T_test     health parameters         (n, 10)
        Y_dev  / Y_test     RUL label                 (n, 1)
        A_dev  / A_test     auxiliary: unit, cycle…   (n, 4)
        W_var, X_s_var, …   the column names for each

    Two things this guards against:

    * MEMORY. A single DS0x is millions of 1 Hz rows; pulling every column of
      every unit into pandas is several GB and will be killed. We read the
      auxiliary array first, pick `max_units` engines, slice only their
      contiguous row block, and take every `stride`-th sample.
    * HONESTY. We use X_s (what a sensor actually measures), not X_v (virtual
      channels the simulator knows but an aircraft does not) and not T (the
      ground-truth health parameters). Training on T would be leakage — it is
      the answer — and would produce a score we could not defend.
    """
    with h5py.File(path, "r") as f:
        a_names = _names(f, "A_var")
        A = np.array(f[f"A_{split}"])                      # (n, 4) — small
        unit_col = a_names.index("unit")
        units = A[:, unit_col]

        chosen = np.unique(units)[:max_units]
        mask_full = np.isin(units, chosen)
        idx = np.flatnonzero(mask_full)
        lo, hi = idx.min(), idx.max() + 1                   # units are contiguous

        # Slice the contiguous block, THEN stride — h5py reads a slab, not
        # millions of scattered rows.
        sl = slice(lo, hi, stride)
        Xs = np.array(f[f"X_s_{split}"][sl])
        W  = np.array(f[f"W_{split}"][sl])
        Y  = np.array(f[f"Y_{split}"][sl]).ravel()
        Ab = A[sl]

        cols = {}
        for j, nm in enumerate(_names(f, "W_var")):
            cols[f"W_{nm}"] = W[:, j]
        for j, nm in enumerate(_names(f, "X_s_var")):
            cols[f"Xs_{nm}"] = Xs[:, j]
        cols["unit"]  = Ab[:, unit_col]
        cols["cycle"] = Ab[:, a_names.index("cycle")]
        cols["RUL"]   = Y

    df = pd.DataFrame(cols)
    df = df[np.isin(df["unit"].values, chosen)].reset_index(drop=True)
    print(f"[{split}] units={sorted(set(df.unit))}  rows={len(df):,}  "
          f"cols={df.shape[1]}  (stride={stride})")
    return df


if NCMAPSS_PATH.exists():
    # dev = development split (train), test = held out. Separate engine units.
    train_df = load_ncmapss(NCMAPSS_PATH, split="dev",  max_units=6, stride=10)
    test_df  = load_ncmapss(NCMAPSS_PATH, split="test", max_units=3, stride=10)
    REAL_DATA = True
else:
    print(f"File not found: {NCMAPSS_PATH}")
    print("Download: NASA PCoE repository — 'Turbofan Engine Degradation "
          "Simulation Data Set 2' (N-CMAPSS).")
    print("Generating synthetic stand-in so the notebook still executes...")
    REAL_DATA = False
    rng = np.random.default_rng(42)
    n_units_train, n_units_test = 50, 15
    records = []
    for u in range(n_units_train + n_units_test):
        eol = rng.integers(100, 400)
        for c in range(eol):
            rul = eol - c - 1
            row = {'unit': u, 'cycle': c, 'RUL': rul,
                   'split': 'train' if u < n_units_train else 'test'}
            for s in range(14):
                row[f'Xs_{s:02d}'] = rng.standard_normal() + 0.1 * (eol - rul) / eol
            records.append(row)
    all_df = pd.DataFrame(records)
    train_df = all_df[all_df.split == 'train'].drop(columns='split').reset_index(drop=True)
    test_df  = all_df[all_df.split == 'test'].drop(columns='split').reset_index(drop=True)
    print(f'Synthetic stand-in: train={len(train_df)}  test={len(test_df)}')

print()
print("REAL_DATA =", REAL_DATA,
      "— any score below is only quotable to judges when this is True.")

## 2. Feature extraction and windowing

In [ ]:
# Identify sensor columns
sensor_cols = [c for c in train_df.columns if c.startswith('X_s_') or c.startswith('X_v_')]
print(f'Using {len(sensor_cols)} sensor features: {sensor_cols}')

def make_windows(df, sensor_cols, window=WINDOW):
    """Sliding windows: (N, T, n_features) and (N,) RUL labels."""
    X_list, y_list = [], []
    for unit, grp in df.groupby('unit'):
        grp = grp.sort_values('cycle')
        feats = grp[sensor_cols].values.astype(np.float32)
        rul   = grp['RUL'].values.astype(np.float32)
        for i in range(window, len(feats)):
            X_list.append(feats[i-window:i])
            y_list.append(rul[i])
    return np.stack(X_list), np.array(y_list)

X_train, y_train = make_windows(train_df, sensor_cols)
X_test,  y_test  = make_windows(test_df,  sensor_cols)
print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}   y_test:  {y_test.shape}')

# Normalise (fit on train, apply to test)
mean = X_train.mean(axis=(0,1), keepdims=True)
std  = X_train.std(axis=(0,1),  keepdims=True) + 1e-8
X_train_n = (X_train - mean) / std
X_test_n  = (X_test  - mean) / std

# Clip RUL at 130 h (standard PHM08 practice — very large RUL values add noise)
RUL_MAX = 130.0
y_train_c = np.clip(y_train, 0, RUL_MAX)
y_test_c  = np.clip(y_test,  0, RUL_MAX)

## 3. Model — GRU with quantile output (same architecture as M3 RUL head)

In [ ]:
class RULPredictor(nn.Module):
    """GRU + quantile regression head (p10/p50/p90)."""
    
    def __init__(self, input_dim, hidden=64, n_layers=2):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden, n_layers, batch_first=True, dropout=0.1)
        self.head = nn.Sequential(
            nn.Linear(hidden, 32),
            nn.ReLU(),
            nn.Linear(32, 3),    # p10, p50, p90
            nn.Softplus()
        )
    
    def forward(self, x):
        # x: (B, T, n_feat)
        out, _ = self.gru(x)          # (B, T, hidden)
        return self.head(out[:, -1])  # (B, 3)


def quantile_loss(pred, target, quantiles=[0.1, 0.5, 0.9]):
    q = torch.tensor(quantiles, device=pred.device)
    target = target.unsqueeze(1).expand_as(pred)
    diff = target - pred
    return torch.where(diff >= 0, q * diff, (q - 1) * diff).mean()


n_feat = X_train_n.shape[2]
model = RULPredictor(n_feat).to(DEVICE)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

## 4. Train

In [ ]:
EPOCHS = 40
BATCH  = 128

train_ds = TensorDataset(
    torch.from_numpy(X_train_n), torch.from_numpy(y_train_c)
)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)

opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        pred = model(xb)
        loss = quantile_loss(pred, yb)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += loss.item() * xb.size(0)
    sched.step()
    avg = total / len(train_ds)
    history.append(avg)
    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS}  loss={avg:.4f}')

plt.plot(history); plt.xlabel('Epoch'); plt.ylabel('Quantile Loss')
plt.title('N-CMAPSS Training Loss'); plt.tight_layout(); plt.show()

## 5. Evaluation — field-standard metrics

In [ ]:
model.eval()
test_ds = TensorDataset(
    torch.from_numpy(X_test_n), torch.from_numpy(y_test_c)
)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

p10s, p50s, p90s, trues = [], [], [], []
with torch.no_grad():
    for xb, yb in test_loader:
        pred = model(xb.to(DEVICE)).cpu().numpy()
        p10s.extend(pred[:, 0]); p50s.extend(pred[:, 1]); p90s.extend(pred[:, 2])
        trues.extend(yb.numpy())

p10s = np.array(p10s); p50s = np.array(p50s); p90s = np.array(p90s)
trues = np.array(trues)
d = p50s - trues    # d = predicted - true: negative = early (conservative), positive = late

# 1. RMSE
rmse = np.sqrt(np.mean(d**2))

# 2. NASA asymmetric score (PHM08 competition metric)
# Early predictions (d < 0) penalised less than late predictions (d > 0)
nasa_score = np.where(d < 0,
    np.exp(-d / 13) - 1,
    np.exp( d / 10) - 1
).sum()

# 3. Prognostic horizon — earliest point where p50 stays within ±α of true RUL
ALPHA = 0.1
within_band = np.abs(d) / (trues + 1e-9) < ALPHA
ph_pct = within_band.mean() * 100.0   # fraction of test windows within band

# 4. α–λ accuracy: fraction within ±α at λ=0.8 (80% of life consumed)
# Approximate with overall within-band fraction
alpha_lambda = within_band.mean()

# 5. False alarm rate proxy: fraction where p10 > true + 20% margin
false_alarm_proxy = np.mean(p10s > trues * 1.20)

print('═' * 55)
print('N-CMAPSS RUL Benchmark Results (p50 head)')
print('═' * 55)
print(f'  RMSE                     = {rmse:.2f} cycles')
print(f'  NASA asymmetric score    = {nasa_score:.1f}  (lower is better)')
print(f'  Within ±{int(ALPHA*100)}% band         = {ph_pct:.1f}%')
print(f'  α–λ accuracy             = {alpha_lambda:.3f}')
print(f'  False alarm proxy        = {false_alarm_proxy:.3f}')
print('═' * 55)
print()
print('NOTE: This pipeline runs on NASA data unchanged — same')
print('architecture as M3, zero PRAMANA-specific tuning.')
print('This is the slide that answers "why trust synthetic data?"')

## 6. Prediction plot

In [ ]:
n_plot = min(500, len(trues))
idx = np.random.choice(len(trues), n_plot, replace=False)
idx_s = np.argsort(trues[idx])

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(range(n_plot), p10s[idx][idx_s], p90s[idx][idx_s],
                alpha=0.3, label='p10–p90', color='steelblue')
ax.plot(range(n_plot), p50s[idx][idx_s], label='p50 (predicted)', color='steelblue')
ax.plot(range(n_plot), trues[idx][idx_s], label='True RUL', color='crimson', linewidth=1.5)
ax.set_xlabel('Sample (sorted by true RUL)')
ax.set_ylabel('Remaining Useful Life (cycles)')
ax.set_title(f'N-CMAPSS RUL — RMSE={rmse:.1f}  NASA score={nasa_score:.0f}')
ax.legend()
plt.tight_layout()
plt.savefig('ncmapss_rul_result.png', dpi=150)
plt.show()
print('Saved: ncmapss_rul_result.png')

## Summary for the slide

Copy these numbers onto the validation slide:

| Metric | Value |
|--------|-------|
| RMSE | *see above* |
| NASA asymmetric score | *see above* |
| Within ±10% band | *see above* |
| α–λ accuracy | *see above* |
| False alarm proxy | *see above* |

**Key talking point:**  
*"Our ML pipeline ran unmodified on NASA's N-CMAPSS dataset — real flight-condition run-to-failure data we did not generate — and achieved [score]. This is the only thing that proves we are not memorising our own simulator."*

**What to add (Day 4 measurement):** false alarm rate per flight hour on 100 healthy PRAMANA flights.